<a href="https://colab.research.google.com/github/Sridipta-Roy/protein-function-active-learning/blob/main/notebooks/09_prepare_demo_artifacts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/Sridipta-Roy/protein-function-active-learning

Cloning into 'protein-function-active-learning'...
remote: Enumerating objects: 119, done.
remote: Counting objects: 100% (41/41), done.
remote: Compressing objects: 100% (37/37), done.
Receiving objects: 100% (119/119), 59.12 MiB | 10.24 MiB/s, done.
remote: Total 119 (delta 4), reused 39 (delta 4), pack-reused 78 (from 2)
Resolving deltas: 100% (17/17), done.
Updating files: 100% (74/74), done.


# 09 - Prepare deployable demo artifacts (ESM-2 35M)

The notebooks use ESM-2 **650M** (1280-dim) for best accuracy. The live Streamlit demo uses ESM-2 **35M** (480-dim) so it fits free-tier hosting (~1 GB RAM). Same pipeline, sized to the deployment budget.

This notebook (run on **Colab with GPU**, like NB05) does three small things:
1. Generate 35M embeddings for all proteins.
2. Train a LogReg classifier on them (the model the app will load).
3. Save the classifier + a label list to `app/artifacts/`.

The app loads the 35M ESM model live to embed a pasted sequence, then this saved
classifier predicts the class. Everything is small enough to deploy free.


In [2]:
!pip install -q torch transformers scikit-learn joblib

In [3]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# For colab
PROJECT_ROOT = Path("/content/protein-function-active-learning")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SRC_DIR       = PROJECT_ROOT / "src"
ARTIFACT_DIR  = PROJECT_ROOT / "app" / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(SRC_DIR))
from embeddings import ESMEmbedder

CLASS_ORDER = ["enzyme", "dna_rna_binding", "receptor", "transporter", "structural", "other"]
RANDOM_STATE = 42
SMALL_MODEL = "facebook/esm2_t12_35M_UR50D"   # 480-dim, deploy-friendly


## 1. Load proteins (dedup, same as other notebooks)

In [4]:
df = pd.read_csv(PROCESSED_DIR / "labeled_dataset.csv")
df = df.drop_duplicates(subset="sequence", keep="first").reset_index(drop=True)
print(f"{len(df)} unique proteins")


8513 unique proteins


## 2. Generate 35M embeddings

In [ ]:
embedder = ESMEmbedder(model_name=SMALL_MODEL, max_length=1024)
print(f"Model: {SMALL_MODEL}  dim={embedder.dim}  device={embedder.device}")

X, accs = embedder.embed_dataframe(df, id_col="accession", seq_col="sequence", batch_size=16)
y = df.set_index("accession").loc[[str(a) for a in accs], "function_class"].to_numpy()
print("Embeddings:", X.shape)


## 3. Train the deployable classifier

Trained on ALL proteins (no held-out split needed here - evaluation already done
in NB06; this is the final model to ship). Reports train-set macro-F1 as a sanity
check only.

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score

clf = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, C=1.0, random_state=RANDOM_STATE),
)
clf.fit(X, y)
train_f1 = f1_score(y, clf.predict(X), average="macro")
print(f"Train macro-F1 (sanity check only): {train_f1:.3f}")


Train macro-F1 (sanity check only): 0.770


## 4. Save artifacts for the app

In [7]:
import joblib, json

joblib.dump(clf, ARTIFACT_DIR / "classifier_esm35m.joblib")
meta = {
    "esm_model": SMALL_MODEL,
    "embedding_dim": int(embedder.dim),
    "classes": list(clf.classes_),
    "train_macro_f1": round(float(train_f1), 4),
}
(ARTIFACT_DIR / "model_meta.json").write_text(json.dumps(meta, indent=2))

print("Saved:")
print(" ", ARTIFACT_DIR / "classifier_esm35m.joblib")
print(" ", ARTIFACT_DIR / "model_meta.json")


Saved:
  /content/protein-function-active-learning/app/artifacts/classifier_esm35m.joblib
  /content/protein-function-active-learning/app/artifacts/model_meta.json
